निम्नलिखित नोटबुक GitHub Copilot Chat द्वारा स्वचालित रूप से उत्पन्न किया गया था और केवल प्रारंभिक सेटअप के लिए है


# प्रॉम्प्ट इंजीनियरिंग का परिचय
प्रॉम्प्ट इंजीनियरिंग प्राकृतिक भाषा प्रसंस्करण कार्यों के लिए प्रॉम्प्ट को डिजाइन और अनुकूलित करने की प्रक्रिया है। इसमें सही प्रॉम्प्ट का चयन करना, उनके पैरामीटर को समायोजित करना, और उनके प्रदर्शन का मूल्यांकन करना शामिल है। प्रॉम्प्ट इंजीनियरिंग एनएलपी मॉडलों में उच्च सटीकता और दक्षता प्राप्त करने के लिए महत्वपूर्ण है। इस खंड में, हम एक्सप्लोरेशन के लिए OpenAI मॉडलों का उपयोग करते हुए प्रॉम्प्ट इंजीनियरिंग के मूल बातें जानेंगे।


### अभ्यास 1: टोकनाइज़ेशन
tiktoken का उपयोग करके टोकनाइज़ेशन का अन्वेषण करें, जो OpenAI का एक ओपन-सोर्स तेज़ टोकनाइज़र है
और अधिक उदाहरणों के लिए [OpenAI कुकबुक](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) देखें।


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### अभ्यास 2: OpenAI API कुंजी सेटअप सत्यापित करें

नीचे दिया गया कोड चलाएं यह सत्यापित करने के लिए कि आपका OpenAI endpoint सही ढंग से सेटअप है। कोड एक सरल बेसिक प्रॉम्प्ट कोशिश करता है और पूरा होने को सत्यापित करता है। इनपुट `oh say can you see` के पूरा होने का परिणाम कुछ इस प्रकार होना चाहिए `by the dawn's early light..`


In [ ]:
# Uses the OpenAI client against the Azure OpenAI (Microsoft Foundry) v1 endpoint
# with the Responses API. See https://aka.ms/openai/start

import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

def get_completion(prompt):
    response = client.responses.create(
        model=deployment,
        input=prompt,
        temperature=0, # this is the degree of randomness of the model's output
        max_output_tokens=1024,
        store=False,
    )
    return response.output_text

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt)
print(response)


### अभ्यास 3: निर्माण
यह पता लगाएं कि जब आप LLM से किसी ऐसे विषय पर प्रॉम्प्ट के लिए पूर्णताएँ वापस मांगते हैं जो मौजूद न हो, या उन विषयों के बारे में जो शायद वह न जानता हो क्योंकि वे इसके पूर्व-प्रशिक्षित डेटासेट (हाल का) के बाहर हैं, तो क्या होता है। देखें कि प्रतिक्रिया कैसे बदलती है यदि आप अलग प्रॉम्प्ट या अलग मॉडल का प्रयास करते हैं।


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt)
print(response)

### अभ्यास 4: निर्देश आधारित 
प्राथमिक सामग्री सेट करने के लिए "text" वेरिएबल का उपयोग करें 
और उस प्राथमिक सामग्री से संबंधित निर्देश प्रदान करने के लिए "prompt" वेरिएबल का उपयोग करें।

यहाँ हम मॉडल से टेक्स्ट को दूसरे ग्रेड के छात्र के लिए संक्षेप में बताने को कहते हैं


In [ ]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### अभ्यास 5: जटिल प्रॉम्प्ट 
एक अनुरोध आज़माएं जिसमें सिस्टम, उपयोगकर्ता और सहायक संदेश शामिल हों 
सिस्टम सहायक संदर्भ सेट करता है
उपयोगकर्ता और सहायक संदेश बहु-चरण वार्तालाप संदर्भ प्रदान करते हैं

ध्यान दें कि सहायक की व्यक्तिमत्ता सिस्टम संदर्भ में "व्यंग्यात्मक" के रूप में सेट है। 
एक अलग व्यक्तिमत्ता संदर्भ आज़माएं। या इनपुट/आउटपुट संदेशों की एक अलग श्रृंखला आज़माएं


In [ ]:
response = client.responses.create(
    model=deployment,
    input=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ],
    store=False,
)
print(response.output_text)


### अभ्यास: अपनी अंतर्ज्ञान का अन्वेषण करें
ऊपर दिए गए उदाहरण आपको नए प्रॉम्प्ट (सरल, जटिल, निर्देश आदि) बनाने के लिए पैटर्न देते हैं - ऐसी अन्य अभ्यास बनाने की कोशिश करें ताकि हम जिन अन्य विचारों के बारे में बात कर चुके हैं जैसे उदाहरण, संकेत और अधिक का अन्वेषण किया जा सके।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
